# STC tv – Task 3: Recommendation System
**الهدف:** بناء نظام توصيات لمنصة stc tv، وعرض أعلى 5 توصيات للأشخاص الذين شاهدوا فيلم **Moana**.

**الطريقة:** Item-Based Collaborative Filtering باستخدام KNN و Cosine Similarity.
الفكرة: البرنامجان متشابهان إذا شاهدهما ونال إعجابَ نفس المستخدمين.

**خطوات العمل:**
1. تحميل البيانات واستكشافها
2. معالجة التكرار (صف واحد لكل مستخدم + برنامج)
3. حذف البرامج قليلة المشاهدين
4. بناء مصفوفة (برنامج × مستخدم)
5. تدريب النموذج
6. دالة التوصية وعرض نتائج Moana

## 0. المكتبات

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

## 1. تحميل البيانات

In [4]:
FILE_PATH = "stc TV Data Set_T3.xlsx"

dataframe = pd.read_excel(FILE_PATH, index_col=0)
df = dataframe.copy()
df.head()

,user_id_maped,program_name,rating,date_,program_genre
0,26138,100 treets,1,2017-05-27,Drama
1,7946,Moana,1,2017-05-21,Animation
2,7418,The Mermaid Princess,1,2017-08-10,Animation
3,19307,The Mermaid Princess,2,2017-07-26,Animation
4,15860,Churchill,2,2017-07-07,Biography


## 2. استكشاف سريع للبيانات

In [5]:
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nUnique users   :", df['user_id_maped'].nunique())
print("Unique programs:", df['program_name'].nunique())
print("\nRating distribution:")
print(df['rating'].value_counts().sort_index())

Shape: (1048575, 5)

Missing values per column:
user_id_maped    0
program_name     0
rating           0
date_            0
program_genre    0
dtype: int64

Unique users   : 11578
Unique programs: 8013

Rating distribution:
rating
1    264293
2    260664
3    261505
4    262113
Name: count, dtype: int64


**ملاحظات:**
- لا توجد قيم ناقصة.
- `rating` من 1 إلى 4 وموزّع تقريباً بالتساوي. المصدر الداعم ينص على استخدام مدة المشاهدة لمعرفة هل أحبّ المستخدم البرنامج، ونفترض أن `rating` يعكس مدة المشاهدة
- كل صف يمثل **مشاهدة** وليس مستخدماً، لذلك يتكرر المستخدم مع نفس البرنامج.

## 3. معالجة التكرار

In [6]:
total_rows = len(df)
full_dups  = df.duplicated().sum()
pair_dups  = df.duplicated(subset=['user_id_maped', 'program_name']).sum()

print("Total rows                          :", total_rows)
print("Fully duplicated rows               :", full_dups)
print("Duplicated (user + program) rows    :", pair_dups)

Total rows                          : 1048575
Fully duplicated rows               : 238474
Duplicated (user + program) rows    : 608338


الرقم الثاني أكبر بكثير من الأول، لأن المستخدم يشاهد نفس البرنامج في أيام مختلفة وبتقييمات مختلفة.
بعضها تكرار متطابق تماماً (غالباً تسجيل مزدوج)، وبعضها مشاهدات منفصلة فعلاً.

**القرار:** نأخذ **أعلى تقييم (max)** لكل (مستخدم، برنامج).
**السبب:** إذا كان التقييم يعكس مدة المشاهدة، فأطول مشاهدة تعبّر عن أعلى مستوى اهتمام وصله المستخدم.

In [7]:
# One row per (user, program) keeping the highest rating
user_prog = (df.groupby(['user_id_maped', 'program_name'])['rating']
               .max()
               .reset_index())

print("Rows after aggregation :", len(user_prog))
print("Expected (total - dups):", total_rows - pair_dups)

Rows after aggregation : 440237
Expected (total - dups): 440237


## 4. حذف البرامج قليلة المشاهدين
برنامج شاهده مستخدمان فقط قد يظهر تشابهاً عالياً مع Moana بمحض الصدفة، فيظهر في التوصيات دون أن يستحق.
نحسب أولاً عدد المستخدمين لكل برنامج.

In [8]:
viewers_per_program = user_prog.groupby('program_name').size()

print("Total programs:", len(viewers_per_program))
for t in [5, 50, 100]:
    print(f"Programs with fewer than {t:>3} viewers:", (viewers_per_program < t).sum())

Total programs: 8013
Programs with fewer than   5 viewers: 1086
Programs with fewer than  50 viewers: 5745
Programs with fewer than 100 viewers: 6975


**القرار:** الحد الأدنى **50 مشاهداً**.
**السبب:** أقل من 5 مشاهدين نتائجه عشوائية. رفع الحد إلى 100 يحذف حوالي نصف البرامج الباقية (يبقى 1,038 برنامجاً)، بينما 50 يحافظ على تنوع جيد (2,268 برنامجاً) مع بيانات كافية لكل برنامج.

In [9]:
MIN_VIEWERS = 50

popular_programs = viewers_per_program[viewers_per_program >= MIN_VIEWERS].index
user_prog = user_prog[user_prog['program_name'].isin(popular_programs)]

print("Programs kept:", user_prog['program_name'].nunique())
print("Rows kept    :", len(user_prog))
print("Moana kept   :", 'Moana' in popular_programs)

Programs kept: 2268
Rows kept    : 343560
Moana kept   : True


## 5. بناء المصفوفة (برنامج × مستخدم)
- كل **صف** = برنامج، وكل **عمود** = مستخدم، والقيمة = التقييم.
- المصفوفة الكاملة ستحتوي على عشرات الملايين من الخلايا أغلبها فارغ (المستخدم لم يشاهد أغلب البرامج)، لذلك نستخدم مصفوفة **sparse** تخزّن القيم الموجودة فقط. هذا ما يمنع انهيار الذاكرة في Colab.
- `astype('category')` يعطي كل مستخدم وكل برنامج رقماً (`cat.codes`) يُستخدم كإحداثي في المصفوفة.

In [10]:
users = user_prog['user_id_maped'].astype('category')
progs = user_prog['program_name'].astype('category')

matrix = csr_matrix(
    (user_prog['rating'], (progs.cat.codes, users.cat.codes))
)
program_names = list(progs.cat.categories)   # row index -> program name

print("Matrix shape (programs x users):", matrix.shape)

Matrix shape (programs x users): (2268, 11532)


## 6. تدريب النموذج
- `metric='cosine'`: يقيس تشابه اتجاه التقييمات بين برنامجين، أي هل يحبهما نفس الناس، بغض النظر عن حجم الفرق في الأرقام.
- `algorithm='brute'`: يقارن كل برنامج بكل البرامج الأخرى. هذا الخيار مناسب مع cosine على بيانات بهذا الحجم.
- `fit` هنا لا يتعلم أوزاناً، بل يجهّز المصفوفة للبحث عن الجيران الأقرب.

In [11]:
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

## 7. دالة التوصية
- `kneighbors` ترجع أقرب البرامج **بما فيها البرنامج نفسه** (مسافته 0)، لذلك نطلب `n + 1` ونتجاوز الأول.
- الدالة ترجع **مسافة** (كلما قلّت زاد التشابه)، فنحوّلها إلى تشابه: `similarity = 1 - distance`.

In [12]:
# Most common genre per program (a few programs have more than one genre)
genre_map = df.groupby('program_name')['program_genre'].agg(lambda s: s.mode()[0])

def recommend(title, n=5):
    if title not in program_names:
        raise ValueError(f"'{title}' is not in the model (maybe below MIN_VIEWERS or misspelled).")

    i = program_names.index(title)
    distances, indices = model.kneighbors(matrix[i], n_neighbors=n + 1)

    names = [program_names[j] for j in indices[0][1:]]   # skip the program itself
    return pd.DataFrame({
        'rank': range(1, n + 1),
        'program': names,
        'genre': [genre_map[p] for p in names],
        'similarity': (1 - distances[0][1:]).round(3)
    })

## 8. النتيجة: أعلى 5 توصيات لمن شاهد Moana

In [13]:
recommend('Moana', n=5)

,rank,program,genre,similarity
0,1,Trolls,Animation,0.589
1,2,Surf's Up : WaveMania,Animation,0.539
2,3,The Mermaid Princess,Animation,0.502
3,4,The Boss Baby,Animation,0.459
4,5,The Jetsons & WWE: Robo-WrestleMania!,Animation,0.444


## 9. ملاحظات ومحدوديات
- **النتائج منطقية:** أغلب التوصيات أفلام أنيميشن وعائلية، وهذا متوقع لجمهور Moana.
- **افتراض التقييم:** اعتمدنا أن `rating` يعكس مدة المشاهدة. لو كان معنى مختلفاً فقد يتغير تفسير النتائج.
- **اختيار max:** يعطي أعلى اهتمام وصله المستخدم، لكن مشاهدة واحدة شاذة قد ترفع التقييم.
- **الحد الأدنى للمشاهدين:** البرامج الأقل من 50 مشاهداً مستبعدة، فلا يمكن التوصية بها أو لها (مشكلة Cold Start).
- **تحسينات ممكنة:** الدمج مع التصنيف (Hybrid)، أو جعل المتغير ثنائياً (أعجبه إذا التقييم ≥ 3)، أو استخدام Matrix Factorization.